# Demo 1a: Forrásrendszerek feltárása és Python natív CDC

BME Adatmérnökség – 2. hét

Ebben a demóban:
1. PostgreSQL forrásrendszer feltérképezése
2. Adatprofiling és minőségellenőrzés
3. Natív Python CDC (logical replication)

In [1]:
import psycopg2
import psycopg2.extras
import json
from datetime import datetime

# PostgreSQL kapcsolat
conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="webshop", user="dataeng", password="dataeng2024"
)
conn.autocommit = True
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
print("Kapcsolódva a webshop adatbázishoz!")

Kapcsolódva a webshop adatbázishoz!


## 1. Séma feltérképezés (Schema Discovery)

In [2]:
# Táblák listázása és sorszám lekérdezés
cur.execute("""
    SELECT schemaname, tablename 
    FROM pg_tables 
    WHERE schemaname = 'public'
    ORDER BY tablename
""")
tables = cur.fetchall()

print("=== Webshop adatbázis táblái ===\n")
for t in tables:
    cur.execute(f"SELECT COUNT(*) as cnt FROM {t['tablename']}")
    cnt = cur.fetchone()['cnt']
    print(f"  {t['tablename']:20s} → {cnt:>6,} sor")

=== Webshop adatbázis táblái ===

  customers            →     30 sor
  events               →     80 sor
  order_items          →    200 sor
  orders               →    100 sor
  products             →     25 sor


In [3]:
# Oszlop részletek lekérdezése
cur.execute("""
    SELECT table_name, column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
    ORDER BY table_name, ordinal_position
""")
cols = cur.fetchall()

current_table = None
for c in cols:
    if c['table_name'] != current_table:
        current_table = c['table_name']
        print(f"\n{'='*50}")
        print(f"  {current_table}")
        print(f"{'='*50}")
    nullable = "NULL" if c['is_nullable'] == 'YES' else "NOT NULL"
    default = f" DEFAULT {c['column_default']}" if c['column_default'] else ""
    print(f"  {c['column_name']:20s} {c['data_type']:20s} {nullable}{default}")


  customers
  customer_id          integer              NOT NULL DEFAULT nextval('customers_customer_id_seq'::regclass)
  name                 character varying    NOT NULL
  email                character varying    NOT NULL
  city                 character varying    NULL
  segment              character varying    NULL DEFAULT 'standard'::character varying
  created_at           timestamp without time zone NULL DEFAULT now()
  updated_at           timestamp without time zone NULL DEFAULT now()

  events
  event_id             bigint               NOT NULL DEFAULT nextval('events_event_id_seq'::regclass)
  event_type           character varying    NOT NULL
  entity_type          character varying    NULL
  entity_id            integer              NULL
  payload              jsonb                NULL
  created_at           timestamp without time zone NULL DEFAULT now()

  order_items
  item_id              integer              NOT NULL DEFAULT nextval('order_items_item_id_seq'::regc

## 2. Adatprofiling

In [4]:
# Customers profiling
print("=== Ügyfelek profiling ===\n")

cur.execute("SELECT segment, COUNT(*) as cnt FROM customers GROUP BY segment ORDER BY cnt DESC")
for row in cur.fetchall():
    print(f"  Szegmens '{row['segment']}': {row['cnt']} ügyfél")

cur.execute("SELECT city, COUNT(*) as cnt FROM customers GROUP BY city ORDER BY cnt DESC LIMIT 5")
print("\nTop 5 város:")
for row in cur.fetchall():
    print(f"  {row['city']}: {row['cnt']} ügyfél")

# Orders profiling
print("\n=== Rendelések profiling ===\n")
cur.execute("SELECT status, COUNT(*) as cnt FROM orders GROUP BY status ORDER BY cnt DESC")
for row in cur.fetchall():
    print(f"  Státusz '{row['status']}': {row['cnt']} rendelés")

cur.execute("""
    SELECT 
        MIN(total_amount) as min_total,
        AVG(total_amount)::numeric(12,2) as avg_total,
        MAX(total_amount) as max_total,
        MIN(order_date)::date as first_order,
        MAX(order_date)::date as last_order
    FROM orders
""")
stats = cur.fetchone()
print(f"\n  Összeg: min={stats['min_total']}, átlag={stats['avg_total']}, max={stats['max_total']}")
print(f"  Időszak: {stats['first_order']} – {stats['last_order']}")

=== Ügyfelek profiling ===

  Szegmens 'standard': 19 ügyfél
  Szegmens 'premium': 8 ügyfél
  Szegmens 'gold': 3 ügyfél

Top 5 város:
  Budapest: 9 ügyfél
  Szeged: 4 ügyfél
  Debrecen: 4 ügyfél
  Pécs: 3 ügyfél
  Győr: 3 ügyfél

=== Rendelések profiling ===

  Státusz 'delivered': 30 rendelés
  Státusz 'confirmed': 27 rendelés
  Státusz 'pending': 15 rendelés
  Státusz 'shipped': 14 rendelés
  Státusz 'cancelled': 14 rendelés

  Összeg: min=12990.00, átlag=579100.50, max=2079940.00
  Időszak: 2025-11-25 – 2026-02-21


In [5]:
# Adatminőségi ellenőrzések
print("=== Adatminőségi ellenőrzések ===\n")

checks = [
    ("NULL email az ügyféltáblában", 
     "SELECT COUNT(*) as cnt FROM customers WHERE email IS NULL"),
    ("Duplikált email címek", 
     "SELECT COUNT(*) - COUNT(DISTINCT email) as cnt FROM customers"),
    ("Rendelés ügyfél nélkül (referencia integritás)", 
     "SELECT COUNT(*) as cnt FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL"),
    ("Negatív rendelési összeg", 
     "SELECT COUNT(*) as cnt FROM orders WHERE total_amount < 0"),
    ("Jövőbeli rendelési dátum", 
     "SELECT COUNT(*) as cnt FROM orders WHERE order_date > now()"),
]

all_passed = True
for name, query in checks:
    cur.execute(query)
    cnt = cur.fetchone()['cnt']
    status = "PASS" if cnt == 0 else "FAIL"
    if cnt > 0:
        all_passed = False
    print(f"  [{status}] {name}: {cnt}")

print(f"\n{'Minden ellenőrzés sikeres!' if all_passed else 'Vannak hibás ellenőrzések!'}")

=== Adatminőségi ellenőrzések ===

  [PASS] NULL email az ügyféltáblában: 0
  [PASS] Duplikált email címek: 0
  [PASS] Rendelés ügyfél nélkül (referencia integritás): 0
  [PASS] Negatív rendelési összeg: 0
  [PASS] Jövőbeli rendelési dátum: 0

Minden ellenőrzés sikeres!


## 3. Pull vs Push: SQL Polling vs CDC

In [ ]:
# Hagyományos megközelítés: SQL polling (pull)
import time

print("=== SQL Polling (Pull megközelítés) ===\n")
print("Utolsó 5 rendelés lekérdezése 3 másodpercenként...\n")

for i in range(3):
    cur.execute("""
        SELECT order_id, customer_id, status, total_amount, order_date
        FROM orders ORDER BY order_date DESC LIMIT 5
    """)
    rows = cur.fetchall()
    print(f"[Poll #{i+1} – {datetime.now().strftime('%H:%M:%S')}]")
    for r in rows:
        print(f"  Order #{r['order_id']}: {r['status']} – {r['total_amount']} Ft")
    print()
    if i < 2:
        time.sleep(2)

print("Hátrány: felesleges lekérdezések, késleltetés, terhelés a forrásrendszeren.")

## 4. Natív Python CDC – Logical Replication

A PostgreSQL WAL (Write-Ahead Log) logical decoding funkciójával valós idejű változáskövetés.

**Fontos**: A `wal_level=logical` már be van állítva a Docker konfigurációban.

In [7]:
# CDC: Logical Replication Slot létrehozása
import psycopg2.extras

# Új kapcsolat a replikációhoz
rep_conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="webshop", user="dataeng", password="dataeng2024",
    connection_factory=psycopg2.extras.LogicalReplicationConnection
)
rep_cur = rep_conn.cursor()

# Meglévő slot törlése (ha pgoutput típusú maradt a korábbi futásból)
cur.execute("SELECT plugin FROM pg_replication_slots WHERE slot_name = 'demo1a_slot'")
existing = cur.fetchone()
if existing:
    print(f"Meglévő slot törlése (plugin: {existing['plugin']})...")
    rep_cur.drop_replication_slot("demo1a_slot")
    print("  Törölve.")

# Új slot létrehozása test_decoding pluginnel
# test_decoding: beépített PostgreSQL plugin → ember-olvasható szöveg kimenet
# pgoutput:      bináris protokoll → Debezium olvassa (Demo 1b)
rep_cur.create_replication_slot("demo1a_slot", output_plugin="test_decoding")
print("Replication slot 'demo1a_slot' létrehozva (test_decoding).")

print("\nSlot információ:")
cur.execute("SELECT slot_name, plugin, active FROM pg_replication_slots WHERE slot_name = 'demo1a_slot'")
for row in cur.fetchall():
    print(f"  Slot: {row['slot_name']}, Plugin: {row['plugin']}, Aktív: {row['active']}")
print("\nPlugin különbségek:")
print("  test_decoding → szöveges, ember-olvasható (demo célra)")
print("  pgoutput      → bináris, Debezium olvassa (produkciós CDC)")

Meglévő slot törlése (plugin: test_decoding)...
  Törölve.
Replication slot 'demo1a_slot' létrehozva (test_decoding).

Slot információ:
  Slot: demo1a_slot, Plugin: test_decoding, Aktív: False

Plugin különbségek:
  test_decoding → szöveges, ember-olvasható (demo célra)
  pgoutput      → bináris, Debezium olvassa (produkciós CDC)


In [ ]:
# Változások végrehajtása és CDC események olvasása
print("=== Változások végrehajtása ===\n")

# 1. INSERT
cur.execute("""
    INSERT INTO customers (name, email, city, segment)
    VALUES ('Teszt Elek', 'teszt.elek@example.com', 'Budapest', 'standard')
    RETURNING customer_id
""")
new_id = cur.fetchone()['customer_id']
print(f"INSERT: Új ügyfél #{new_id}")

# 2. UPDATE
cur.execute(f"UPDATE customers SET segment = 'premium' WHERE customer_id = {new_id}")
print(f"UPDATE: Ügyfél #{new_id} szegmens → premium")

# 3. DELETE
cur.execute(f"DELETE FROM customers WHERE customer_id = {new_id}")
print(f"DELETE: Ügyfél #{new_id} törölve")

# CDC stream olvasása – test_decoding szöveges formátum
# Kimenet: "table public.customers: INSERT: customer_id[integer]:31 name[text]:'Teszt Elek' ..."
print("\n=== CDC események a WAL-ból ===\n")
rep_cur.start_replication(slot_name="demo1a_slot", decode=True)

class CDCConsumer:
    def __init__(self, max_lines=30):
        self.line_count = 0
        self.max_lines = max_lines

    def __call__(self, msg):
        payload = msg.payload.strip()
        self.line_count += 1

        # Formázott megjelenítés típus szerint
        if payload.startswith("BEGIN"):
            print(f"  ┌─ {payload}")
        elif payload.startswith("COMMIT"):
            print(f"  └─ {payload}\n")
        else:
            # "table public.customers: INSERT: ..." → csak a lényeg
            print(f"  │  {payload[:200]}")

        msg.cursor.send_feedback(flush_lsn=msg.data_start)
        if self.line_count >= self.max_lines:
            raise StopIteration()

consumer = CDCConsumer(max_lines=30)
try:
    rep_cur.consume_stream(consumer, keepalive_interval=1)
except StopIteration:
    print(f"CDC stream feldolgozva ({consumer.line_count} sor).")
except Exception as e:
    print(f"Stream olvasás vége: {e}")

print("\nEz a natív CDC megközelítés. Ipari környezetben Debezium-ot használunk (→ Demo 1b).")

=== Változások végrehajtása ===

INSERT: Új ügyfél #36
UPDATE: Ügyfél #36 szegmens → premium
DELETE: Ügyfél #36 törölve

=== CDC események a WAL-ból ===

  ┌─ BEGIN 785
  │  table public.customers: INSERT: customer_id[integer]:36 name[character varying]:'Teszt Elek' email[character varying]:'teszt.elek@example.com' city[character varying]:'Budapest' segment[character vary
  └─ COMMIT 785

  ┌─ BEGIN 786
  │  table public.customers: UPDATE: customer_id[integer]:36 name[character varying]:'Teszt Elek' email[character varying]:'teszt.elek@example.com' city[character varying]:'Budapest' segment[character vary
  └─ COMMIT 786

  ┌─ BEGIN 787
  │  table public.customers: DELETE: customer_id[integer]:36
  └─ COMMIT 787

  ┌─ BEGIN 789
  │  table public.customers: INSERT: customer_id[integer]:37 name[character varying]:'Demo Béla' email[character varying]:'demo.bela@example.com' city[character varying]:'Debrecen' segment[character varyin
  └─ COMMIT 789

  ┌─ BEGIN 790
  │  table public.custom

In [ ]:
# Takarítás
try:
    rep_cur.drop_replication_slot("demo1a_slot")
    print("Replication slot 'demo1a_slot' törölve.")
except:
    pass

rep_conn.close()
cur.close()
conn.close()
print("Kapcsolatok lezárva.")
print("\n=== Demo 1a vége ===")